In [5]:
import datasets

In [6]:
# ds = load_dataset("davidstap/biblenlp-corpus-mmteb", 'eng-deu')
ds = datasets.load_dataset("json", data_files="./data/en_kjv.json", split="train")
ds

Dataset({
    features: ['abbrev', 'name', 'chapters'],
    num_rows: 66
})

In [7]:
len(ds[0]["chapters"][0])

31

In [8]:
df = ds.to_pandas()
df

,abbrev,name,chapters
0,gn,Gênesis,[[In the beginning God created the heaven and ...
1,ex,Êxodo,[[Now these are the names of the children of I...
2,lv,Levítico,"[[And the LORD called unto Moses, and spake un..."
3,nm,Números,[[And the LORD spake unto Moses in the wildern...
4,dt,Deuteronômio,[[These be the words which Moses spake unto al...
...,...,...,...
61,1jo,1 João,"[[That which was from the beginning, which we ..."
62,2jo,2 João,[[The elder unto the elect lady and her childr...
63,3jo,3 João,"[[The elder unto the wellbeloved Gaius, whom I..."
64,jd,Judas,"[[Jude, the servant of Jesus Christ, and broth..."


In [9]:
# [doc for doc in ds]

In [10]:
from langchain_core.documents import Document as LangchainDocument
from langchain_text_splitters import RecursiveCharacterTextSplitter

docs = []

for book in ds:
    chapters = []
    chapter_index = 0
    joined_chapter = ""
    for chapter in book["chapters"]:
        joined_verse = "\n".join(chapter)
        # chapters.append(joined_verse)
        chapter_index += 1

        # for _chapter in chapters:
        joined_chapter += f"--CHAPTER {chapter_index}--\n" + joined_verse

    docs.append(
        LangchainDocument(
            page_content=joined_chapter, metadata={"chapter": book["name"]}
        )
    )

docs[0]


# langchain_docs = [LangchainDocument(page_content='\'.join() '--CHAPTER--'.join() for chapter in book , metadata={"chapter": book["name"]}) for book in ds]
# langchain_docs

Document(metadata={'chapter': 'Gênesis'}, page_content='--CHAPTER 1--\nIn the beginning God created the heaven and the earth.\nAnd the earth was without form, and void; and darkness was upon the face of the deep. And the Spirit of God moved upon the face of the waters.\nAnd God said, Let there be light: and there was light.\nAnd God saw the light, that it was good: and God divided the light from the darkness.\nAnd God called the light Day, and the darkness he called Night. And the evening and the morning were the first day.\nAnd God said, Let there be a firmament in the midst of the waters, and let it divide the waters from the waters.\nAnd God made the firmament, and divided the waters which were under the firmament from the waters which were above the firmament: and it was so.\nAnd God called the firmament Heaven. And the evening and the morning were the second day.\nAnd God said, Let the waters under the heaven be gathered together unto one place, and let the dry land appear: and it

In [11]:
from langchain_core.documents import Document as LangchainDocument

docs_as_mark = ""

for book in ds:
    print(book["name"])
    if book["name"] == "Êxodo":
        break
    chapter_index = 0
    joined_chapter = ""
    for chapter in book["chapters"]:
        joined_verse = ""
        for index, verse in enumerate(chapter):
            joined_verse += f"### V{index + 1}\n" + verse + "\n"

        # chapters.append(joined_verse)
        chapter_index += 1

        # for _chapter in chapters:
        joined_chapter += f"## C{chapter_index}\n\n" + joined_verse + "\n\n"

    docs_as_mark += f"# {book['name']}\n\n" + joined_chapter + "\n\n"

print(docs_as_mark)

Gênesis
Êxodo
# Gênesis

## C1

### V1
In the beginning God created the heaven and the earth.
### V2
And the earth was without form, and void; and darkness was upon the face of the deep. And the Spirit of God moved upon the face of the waters.
### V3
And God said, Let there be light: and there was light.
### V4
And God saw the light, that it was good: and God divided the light from the darkness.
### V5
And God called the light Day, and the darkness he called Night. And the evening and the morning were the first day.
### V6
And God said, Let there be a firmament in the midst of the waters, and let it divide the waters from the waters.
### V7
And God made the firmament, and divided the waters which were under the firmament from the waters which were above the firmament: and it was so.
### V8
And God called the firmament Heaven. And the evening and the morning were the second day.
### V9
And God said, Let the waters under the heaven be gathered together unto one place, and let the dry lan

In [12]:
total_chars = sum(len(doc.page_content) for doc in docs)
print(f"Total characters: {total_chars}")
# print(docs[0].page_content[:500])

Total characters: 4160598


In [13]:
# chunk document based spliter using chapter then using text based spliter using resursive split

# Foo

## Bar

Hi this is Jim  
Hi this is Joe

 ## Baz
 
Hi this is Molly'

In [14]:
# MD splits
from langchain_text_splitters import MarkdownHeaderTextSplitter

headers_to_split_on_chapter = [
    ("#", "Book"),
    ("##", "Chapter"),
]

markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on_chapter)
chapter_splits = markdown_splitter.split_text(docs_as_mark)
chapter_splits

[Document(metadata={'Book': 'Gênesis', 'Chapter': 'C1'}, page_content='### V1\nIn the beginning God created the heaven and the earth.\n### V2\nAnd the earth was without form, and void; and darkness was upon the face of the deep. And the Spirit of God moved upon the face of the waters.\n### V3\nAnd God said, Let there be light: and there was light.\n### V4\nAnd God saw the light, that it was good: and God divided the light from the darkness.\n### V5\nAnd God called the light Day, and the darkness he called Night. And the evening and the morning were the first day.\n### V6\nAnd God said, Let there be a firmament in the midst of the waters, and let it divide the waters from the waters.\n### V7\nAnd God made the firmament, and divided the waters which were under the firmament from the waters which were above the firmament: and it was so.\n### V8\nAnd God called the firmament Heaven. And the evening and the morning were the second day.\n### V9\nAnd God said, Let the waters under the heaven 

In [15]:
# MD splits
from langchain_text_splitters import MarkdownHeaderTextSplitter

headers_to_split_on_verse = [
    ("#", "Book"),
    ("##", "Chapter"),
    ("###", "Verse"),
]

markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on_verse)
verse_splits = markdown_splitter.split_text(docs_as_mark)
verse_splits

[Document(metadata={'Book': 'Gênesis', 'Chapter': 'C1', 'Verse': 'V1'}, page_content='In the beginning God created the heaven and the earth.'),
 Document(metadata={'Book': 'Gênesis', 'Chapter': 'C1', 'Verse': 'V2'}, page_content='And the earth was without form, and void; and darkness was upon the face of the deep. And the Spirit of God moved upon the face of the waters.'),
 Document(metadata={'Book': 'Gênesis', 'Chapter': 'C1', 'Verse': 'V3'}, page_content='And God said, Let there be light: and there was light.'),
 Document(metadata={'Book': 'Gênesis', 'Chapter': 'C1', 'Verse': 'V4'}, page_content='And God saw the light, that it was good: and God divided the light from the darkness.'),
 Document(metadata={'Book': 'Gênesis', 'Chapter': 'C1', 'Verse': 'V5'}, page_content='And God called the light Day, and the darkness he called Night. And the evening and the morning were the first day.'),
 Document(metadata={'Book': 'Gênesis', 'Chapter': 'C1', 'Verse': 'V6'}, page_content='And God said, 

In [16]:
# Char-level splits

chunk_size = 100
chunk_overlap = 20
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="gpt2",
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    separators=["\n-"],
)
# Split
chapter_r_splits = text_splitter.split_documents(chapter_splits)
chapter_r_splits

[Document(metadata={'Book': 'Gênesis', 'Chapter': 'C1'}, page_content='### V1\nIn the beginning God created the heaven and the earth.\n### V2\nAnd the earth was without form, and void; and darkness was upon the face of the deep. And the Spirit of God moved upon the face of the waters.\n### V3\nAnd God said, Let there be light: and there was light.\n### V4\nAnd God saw the light, that it was good: and God divided the light from the darkness.\n### V5\nAnd God called the light Day, and the darkness he called Night. And the evening and the morning were the first day.\n### V6\nAnd God said, Let there be a firmament in the midst of the waters, and let it divide the waters from the waters.\n### V7\nAnd God made the firmament, and divided the waters which were under the firmament from the waters which were above the firmament: and it was so.\n### V8\nAnd God called the firmament Heaven. And the evening and the morning were the second day.\n### V9\nAnd God said, Let the waters under the heaven 

In [17]:
from pathlib import Path
import bm25s
import Stemmer
import re

In [18]:
index_dir = Path("indices") / Path("bm25s_indices") / "bible_kjv"
index_dir.mkdir(parents=True, exist_ok=True)

corpus_record = [
    {"id": i, "document": doc.page_content, **doc.metadata}
    for i, doc in enumerate(verse_splits)
]
corpus_lst = [
    doc.page_content for doc in verse_splits
]  # Use the first 20 documents for demonstration

print(f"We have {len(corpus_lst)} documents in the corpus.")

stemmer = Stemmer.Stemmer("english")
tokenizer = bm25s.tokenization.Tokenizer(
    stemmer=stemmer, splitter=lambda text: re.findall(r"\b\w+\b", text)
)
corpus_tokens = tokenizer.tokenize(corpus_lst, return_as="tuple")

retriever = bm25s.BM25(corpus=corpus_record)
retriever.index(corpus_tokens)

retriever.save(index_dir)
tokenizer.save_vocab(index_dir)
tokenizer.save_stopwords(index_dir)
print(f"Saved the index to {index_dir}.")

# get memory usage
mem_use = bm25s.utils.benchmark.get_max_memory_usage()
print(f"Peak memory usage: {mem_use:.2f} GB")

We have 1533 documents in the corpus.


Saved the index to indices/bm25s_indices/bible_kjv.
Peak memory usage: 1.30 GB


In [19]:
import bm25s
from bm25s.tokenization import Tokenizer
import Stemmer

reloaded_retriever = bm25s.BM25.load(
    "indices/bm25s_indices/bible_kjv", load_corpus=True
)
reloaded_tokenizer = Tokenizer(
    stemmer=Stemmer.Stemmer("english"),
    splitter=lambda text: re.findall(r"\b\w+\b", text),
)
reloaded_tokenizer.load_vocab("indices/bm25s_indices/bible_kjv")

In [20]:
# 3. Search!
queries = ["what did God create in the beginning?"]

In [29]:
# -- explaination ---
# Each item in corpus_lst is one verse. The tokenizer finds words, removes common words, and reduces words to their base form.
#  The tokenizer finds words with re.findall(r"\b\w+\b", text) rather than splitting only on spaces.
#  It lowercases them, removes common English words by default, and stems words, so create and created can match through a shared stem.
# The query uses the same tokenizer. BM25 compares its terms with the terms in each verse.
# BM25 gives more weight to rare terms. It also uses term count and verse length to set the score.
# A term gets more weight when it is rare across the corpus; its frequency within a verse and the verse’s length also affect the score.
# The first verse (“In the beginning God created the heaven and the earth”) ranks highest because it matches 'beginning', 'God', and 'create'.
# The code shows the 10 verses with the highest scores.

# Query the corpus
query_tokens = reloaded_tokenizer.tokenize(queries, update_vocab=False)

# Get top-k results as a tuple of (doc ids, scores). Both are arrays of shape (n_queries, k).
# To return docs instead of IDs, set the `corpus=corpus` parameter.
results, scores = reloaded_retriever.retrieve(query_tokens, k=10)

for i in range(results.shape[1]):
    doc, score = results[0, i], scores[0, i]
    print(f"Rank {i + 1} (score: {score:.2f}): {doc}")

Rank 1 (score: 7.37): {'id': 0, 'document': 'In the beginning God created the heaven and the earth.', 'Book': 'Gênesis', 'Chapter': 'C1', 'Verse': 'V1'}
Rank 2 (score: 4.57): {'id': 26, 'document': 'So God created man in his own image, in the image of God created he him; male and female created he them.', 'Book': 'Gênesis', 'Chapter': 'C1', 'Verse': 'V27'}
Rank 3 (score: 3.62): {'id': 106, 'document': 'This is the book of the generations of Adam. In the day that God created man, in the likeness of God made he him;', 'Book': 'Gênesis', 'Chapter': 'C5', 'Verse': 'V1'}
Rank 4 (score: 3.50): {'id': 68, 'document': 'And the LORD God said unto the woman, What is this that thou hast done? And the woman said, The serpent beguiled me, and I did eat.', 'Book': 'Gênesis', 'Chapter': 'C3', 'Verse': 'V13'}
Rank 5 (score: 3.28): {'id': 34, 'document': 'These are the generations of the heavens and of the earth when they were created, in the day that the LORD God made the earth and the heavens,', 'Boo

In [34]:
from sentence_transformers import SentenceTransformer

model_id = "google/embeddinggemma-2"
model = SentenceTransformer(
    model_id, config_kwargs={"vision_config": None, "audio_config": None}
)

# print(f"device: {model.device}")
# print(f"dtype: {model.dtype}")
# print(model)
# print(
#     "Total number of parameters in the model:",
#     sum(p.numel() for p in model.parameters()),
# )

Loading weights: 100%|██████████| 413/413 [00:00<00:00, 7438.50it/s]


In [ ]:
# Define a minimum confidence threshold for a match to be considered valid
similarity_threshold = 0.4  # @param {"type":"slider","min":0,"max":1,"step":0.1}

# -- explanation --
# The model converts the query and each verse into embeddings. An embedding is a list of numbers that represents text.
# It compares each verse embedding with the query embedding and gives each pair a similarity score.
# A high score means that the texts are closely related. The words do not have to match exactly.
# BM25 scores matching words. This model can also find verses that use different words with similar meaning.
# This can help with paraphrased queries. BM25 can work better when the query has exact names or phrases.
# If the best score is below 0.4, the code shows no result. Otherwise, it shows up to 10 verses with the highest scores above 0.4.

query_embedding = model.encode_query(queries, prompt_name="Retrieval-query")

# Encode the document for similarity check
doc_texts = [f"title: 'none' | text: {_verse}" for _verse in corpus_lst]
doc_embeddings = model.encode(doc_texts)

similarities = model.similarity(query_embedding, doc_embeddings)[0]

print(similarities)
condition = similarities is not None and similarities.nelement() != 0

# Find the index and value of the highest score
best_document_index = similarities.argmax().item() if condition else None
print(best_document_index)
document_score = similarities[best_document_index] if condition else 0.0

# Check if the document score meets the threshold
if document_score < similarity_threshold:
    print(
        f" `-> 🤷 No relevant document found. The highest score was only {document_score:.2f}."
    )
else:
    import torch

    values, indices = torch.topk(similarities, k=10)
    for value, index in zip(values, indices):
        if value < similarity_threshold:
            continue
        print(f" `-> ✅ Document Found: '{corpus_lst[index]}' (Score: {value:.2f})")

    # best_document = corpus_lst[best_document_index]
    # 3. Display the final successful result
    # print(f" `-> ✅ Document Found: '{best_document}' (Score: {document_score:.2f})")

tensor([0.8452, 0.7154, 0.7373,  ..., 0.6738, 0.6517, 0.6114])
0
 `-> ✅ Document Found: 'In the beginning God created the heaven and the earth.' (Score: 0.85)
 `-> ✅ Document Found: 'These are the generations of the heavens and of the earth when they were created, in the day that the LORD God made the earth and the heavens,' (Score: 0.78)
 `-> ✅ Document Found: 'And the LORD God planted a garden eastward in Eden; and there he put the man whom he had formed.' (Score: 0.78)
 `-> ✅ Document Found: 'And God called the light Day, and the darkness he called Night. And the evening and the morning were the first day.' (Score: 0.77)
 `-> ✅ Document Found: 'Male and female created he them; and blessed them, and called their name Adam, in the day when they were created.' (Score: 0.77)
 `-> ✅ Document Found: 'And God said, Let the earth bring forth the living creature after his kind, cattle, and creeping thing, and beast of the earth after his kind: and it was so.' (Score: 0.77)
 `-> ✅ Document Fo